In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import joblib

import os

In [3]:
route_dataset_df = pd.read_pickle(
    "../data/processed/route_dataset.pkl"
)

print(route_dataset_df.head())
print()
print(route_dataset_df.shape)

   Route ID  Driver ID  Country  Week ID Day of Week  \
0         0          0        1        0      Monday   
1         1          1        1        0      Monday   
2         2          2        1        0      Monday   
3         3          3        1        0      Monday   
4         4          4        1        0      Monday   

                             Planned Route  \
0                    [0, 1, 2, 3, 3, 0, 1]   
1                    [0, 4, 5, 6, 7, 8, 9]   
2              [0, 10, 11, 12, 13, 14, 15]   
3  [0, 16, 17, 18, 19, 20, 21, 22, 20, 23]   
4          [0, 24, 25, 26, 27, 28, 29, 30]   

                              Actual Route  Planned Stop Count  \
0                    [0, 3, 3, 0, 1, 2, 1]                   7   
1                    [0, 9, 4, 7, 6, 5, 8]                   7   
2              [0, 12, 13, 11, 10, 14, 15]                   7   
3  [0, 16, 19, 17, 18, 20, 22, 20, 21, 23]                  10   
4          [0, 24, 30, 26, 27, 28, 29, 25]              

In [4]:
day_mapping = {
    "Monday": 0,
    "Tuesday": 1,
    "Wednesday": 2,
    "Thursday": 3,
    "Friday": 4,
    "Saturday": 5,
    "Sunday": 6
}

route_dataset_df["Day of Week"] = (
    route_dataset_df["Day of Week"]
    .map(day_mapping)
)

label_mapping = {
    "ND": 0,
    "D": 1
}

route_dataset_df["Label"] = (
    route_dataset_df["Label"]
    .map(label_mapping)
)

print(route_dataset_df[["Day of Week", "Label"]].head())

   Day of Week  Label
0            0      1
1            0      1
2            0      1
3            0      1
4            0      1


In [5]:
feature_columns = [
    "Driver ID",
    "Country",
    "Week ID",
    "Day of Week",
    "Planned Stop Count",
    "Planned Distance"
]

X = route_dataset_df[
    feature_columns
]

y = route_dataset_df[
    "Label"
]

print(X.head())

print()
print(y.head())

   Driver ID  Country  Week ID  Day of Week  Planned Stop Count  \
0          0        1        0            0                   7   
1          1        1        0            0                   7   
2          2        1        0            0                   7   
3          3        1        0            0                  10   
4          4        1        0            0                   8   

   Planned Distance  
0         49.468094  
1         33.274342  
2         12.124804  
3         19.039848  
4         20.632674  

0    1
1    1
2    1
3    1
4    1
Name: Label, dtype: int64


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Shape :", X_train.shape)
print("Testing Shape  :", X_test.shape)

print()
print(y_train.value_counts(normalize=True))

Training Shape : (15717, 6)
Testing Shape  : (3930, 6)

Label
1    0.61882
0    0.38118
Name: proportion, dtype: float64


In [7]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(
    X_train,
    y_train
)

print("Model training completed.")

Model training completed.


In [8]:
y_pred = rf_model.predict(
    X_test
)

print(y_pred[:10])

[0 0 0 1 1 1 0 1 0 0]


In [9]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print(
    f"Accuracy : {accuracy:.4f}"
)

Accuracy : 0.7700


In [10]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.71      0.66      0.69      1498
           1       0.80      0.84      0.82      2432

    accuracy                           0.77      3930
   macro avg       0.76      0.75      0.75      3930
weighted avg       0.77      0.77      0.77      3930



In [11]:
cm = confusion_matrix(
    y_test,
    y_pred
)

print("Confusion Matrix:\n")
print(cm)

Confusion Matrix:

[[ 988  510]
 [ 394 2038]]


In [12]:
importance_df = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": rf_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print(importance_df)

              Feature  Importance
5    Planned Distance    0.258677
0           Driver ID    0.249877
4  Planned Stop Count    0.183409
2             Week ID    0.162475
3         Day of Week    0.075070
1             Country    0.070492


In [13]:
os.makedirs(
    "../outputs/models",
    exist_ok=True
)

joblib.dump(
    rf_model,
    "../outputs/models/random_forest_classifier.pkl"
)

print(
    "Random Forest model saved successfully!"
)

Random Forest model saved successfully!


In [14]:
X_train.to_csv(
    "../outputs/results/X_train.csv",
    index=False
)

X_test.to_csv(
    "../outputs/results/X_test.csv",
    index=False
)

print(
    "Train/Test datasets saved."
)

Train/Test datasets saved.
